[← Volver al índice del curso](../../../INDICE_CURSO.md) · [Guía del tema 07](README.md)

# Híbrido MPI + OpenMP

**Tema:** 07 · **Sesiones:** 31, 34 · **Edición:** 1.0.2026

**Pregunta guía:** ¿Cómo mapear procesos e hilos a nodos, NUMA y núcleos sin sobresuscripción?


## Resultados de aprendizaje

- Relacionar ranks, hilos, núcleos y dominios NUMA.
- Interpretar niveles de `MPI_Init_thread`.
- Diseñar afinidad y first-touch reproducibles.


## Modelo conceptual

MPI distribuye memoria entre procesos; OpenMP explota memoria compartida dentro del proceso.

El nivel de soporte de hilos limita qué hilos pueden invocar MPI.

ranks × threads debe corresponder a la asignación y la afinidad debe evitar migraciones no controladas.


In [ ]:
from pathlib import Path

def find_repository(start: Path) -> Path:
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / "INDICE_CURSO.md").is_file():
            return candidate
    raise RuntimeError("No se encontró la raíz del repositorio")

ROOT = find_repository(Path.cwd())
TOPIC = "07"
NOTEBOOK = "07_hibrido/01_mpi_openmp.ipynb"
assert (ROOT / "curso" / "notebooks" / "07_hibrido" / "README.md").is_file()
print(f"Repositorio: {ROOT}")
print(f"Notebook: {NOTEBOOK}")


## Mapa de recursos

Se genera una asignación simple por nodo y se comprueba que no exceda núcleos.


In [ ]:
nodes, cores_per_node, ranks_per_node, threads_per_rank = 2, 32, 4, 8
assert ranks_per_node * threads_per_rank <= cores_per_node
mapping = []
for node in range(nodes):
    for local_rank in range(ranks_per_node):
        first_core = local_rank * threads_per_rank
        mapping.append((node, local_rank, tuple(range(first_core, first_core+threads_per_rank))))
for row in mapping: print(row)


**Interpretación.** En hardware con SMT o NUMA, la política se adapta y se registra mediante herramientas del runtime/planificador.


## Soporte de hilos MPI

Se ordenan los niveles y se verifica una solicitud.


In [ ]:
levels = {"SINGLE": 0, "FUNNELED": 1, "SERIALIZED": 2, "MULTIPLE": 3}
requested, provided = "FUNNELED", "SERIALIZED"
assert levels[provided] >= levels[requested]
for name, value in levels.items(): print(value, name)
print("solicitado", requested, "provisto", provided)


**Interpretación.** El programa aborta o cambia de estrategia si el nivel provisto es inferior al solicitado.


## Práctica reproducible

1. Elegir ranks por NUMA y hilos por rank.
2. Registrar `OMP_PLACES`, `OMP_PROC_BIND` y opciones Slurm.
3. Comparar MPI puro, OpenMP puro e híbrido con igual total de núcleos.


## Errores frecuentes

- Usar `MPI_THREAD_MULTIPLE` sin necesitarlo ni medir costo.
- Olvidar que bibliotecas internas pueden crear hilos.
- Comparar configuraciones con distinto total de recursos.

## Criterios de aceptación

- No hay sobresuscripción involuntaria.
- Nivel MPI provisto comprobado.
- Afinidad y first-touch documentados.


## Referencias y material relacionado

- [Entornos de clúster](../../../topicos_avanzados/ENTORNOS_CLUSTER.md)
- [Planeación híbrida](../../../docs/PLANEACION_CURSO.md)


[← Volver al índice del curso](../../../INDICE_CURSO.md) · [Continuar desde la guía del tema 07](README.md)
